# Day 040 | K-Nearest Neighbors (KNN)
## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** 2026-05-16  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| KNN algorithm | Classify by majority vote of nearest neighbors | Classify soil type from lab test results |
| Distance metrics | How "closeness" is measured | Euclidean vs Manhattan distance between soil samples |
| Effect of K | Number of neighbors to consider | K=1 overfits, K=50 underfits |
| Feature scaling | Normalize inputs to same range | N-value (0-60) vs water content (0-100%) |
| Decision boundaries | Regions assigned to each class | Mapping soil type zones in feature space |

---

## Our Geotechnical Problem

A geotechnical engineer has collected **100 soil samples** from a site investigation. Each sample has lab test results:
- **N-value** (SPT blow count)
- **Fines content** (%)
- **Water content** (%)
- **Liquid limit** (%)
- **Plasticity index** (%)

The goal is to **automatically classify the soil type** (Clay, Sand, Silt, or Gravel) from these properties using KNN.

KNN is one of the simplest ML algorithms: to classify a new sample, it finds the K most similar samples in the training data and assigns the majority class. No model is "trained" in the traditional sense — the algorithm simply memorizes the data and compares at prediction time.

---

## Part 1: Generate Synthetic Soil Classification Data

We create realistic soil samples where each soil type has characteristic property ranges:
- **Clay**: High fines, high water content, high plasticity
- **Sand**: Low fines, low water content, low plasticity
- **Silt**: Moderate fines, moderate water content, low-moderate plasticity
- **Gravel**: Very low fines, very low water content, zero plasticity

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib.colors import ListedColormap

np.random.seed(42)
print("All imports successful.")

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATE SYNTHETIC SOIL DATA
# ─────────────────────────────────────────────────────

n_per_class = 25  # 25 samples per soil type = 100 total

# Clay: high fines, high water content, high plasticity
clay = pd.DataFrame({
    'N_value': np.random.randint(2, 12, n_per_class),
    'fines_content': np.round(np.random.uniform(70, 98, n_per_class), 1),
    'water_content': np.round(np.random.uniform(25, 55, n_per_class), 1),
    'liquid_limit': np.round(np.random.uniform(35, 70, n_per_class), 1),
    'plasticity_index': np.round(np.random.uniform(15, 40, n_per_class), 1),
    'soil_type': 'Clay'
})

# Sand: low fines, low water content, low plasticity
sand = pd.DataFrame({
    'N_value': np.random.randint(10, 45, n_per_class),
    'fines_content': np.round(np.random.uniform(2, 15, n_per_class), 1),
    'water_content': np.round(np.random.uniform(5, 18, n_per_class), 1),
    'liquid_limit': np.round(np.random.uniform(0, 20, n_per_class), 1),
    'plasticity_index': np.round(np.random.uniform(0, 5, n_per_class), 1),
    'soil_type': 'Sand'
})

# Silt: moderate fines, moderate water content, low-moderate plasticity
silt = pd.DataFrame({
    'N_value': np.random.randint(4, 20, n_per_class),
    'fines_content': np.round(np.random.uniform(50, 85, n_per_class), 1),
    'water_content': np.round(np.random.uniform(18, 35, n_per_class), 1),
    'liquid_limit': np.round(np.random.uniform(20, 40, n_per_class), 1),
    'plasticity_index': np.round(np.random.uniform(2, 12, n_per_class), 1),
    'soil_type': 'Silt'
})

# Gravel: very low fines, low water content, zero plasticity
gravel = pd.DataFrame({
    'N_value': np.random.randint(25, 60, n_per_class),
    'fines_content': np.round(np.random.uniform(0, 8, n_per_class), 1),
    'water_content': np.round(np.random.uniform(2, 10, n_per_class), 1),
    'liquid_limit': np.round(np.random.uniform(0, 10, n_per_class), 1),
    'plasticity_index': np.round(np.random.uniform(0, 2, n_per_class), 1),
    'soil_type': 'Gravel'
})

# Combine all soil types
df = pd.concat([clay, sand, silt, gravel], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

# Save to CSV
df.to_csv('soil_classification.csv', index=False)
print(f"Dataset shape: {df.shape}")
print(f"\nSoil type distribution:\n{df['soil_type'].value_counts()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# ─────────────────────────────────────────────────────
# EXPLORE THE DATA
# ─────────────────────────────────────────────────────

print("Statistical summary by soil type:")
print("=" * 60)
for soil in ['Clay', 'Sand', 'Silt', 'Gravel']:
    subset = df[df['soil_type'] == soil]
    print(f"\n{soil} (n={len(subset)}):")
    print(f"  N-value:         {subset['N_value'].mean():5.1f} +/- {subset['N_value'].std():4.1f}")
    print(f"  Fines content:   {subset['fines_content'].mean():5.1f}% +/- {subset['fines_content'].std():4.1f}")
    print(f"  Water content:   {subset['water_content'].mean():5.1f}% +/- {subset['water_content'].std():4.1f}")
    print(f"  Liquid limit:    {subset['liquid_limit'].mean():5.1f}% +/- {subset['liquid_limit'].std():4.1f}")
    print(f"  Plasticity idx:  {subset['plasticity_index'].mean():5.1f}% +/- {subset['plasticity_index'].std():4.1f}")

---

## Part 2: Understanding KNN — How It Works

KNN is an **instance-based** (lazy) learner:

1. **Store** all training samples in memory
2. When a new sample arrives, **compute distance** to every training sample
3. Find the **K closest** neighbors
4. **Vote**: the majority class among those K neighbors becomes the prediction

**Distance Metrics:**
- **Euclidean**: Straight-line distance: $d = \sqrt{\sum(x_i - y_i)^2}$
- **Manhattan**: Sum of absolute differences: $d = \sum|x_i - y_i|$

**Why Feature Scaling Matters:**
If N-value ranges from 0-60 but fines content ranges from 0-100, fines content will dominate the distance calculation. We must scale all features to the same range.

In [ ]:
# ─────────────────────────────────────────────────────
# DISTANCE METRICS DEMONSTRATION
# ─────────────────────────────────────────────────────

# Two soil samples
sample_a = np.array([15, 45, 22])  # [N_value, fines_content, water_content]
sample_b = np.array([25, 10, 8])

# Euclidean distance
euclidean = np.sqrt(np.sum((sample_a - sample_b) ** 2))

# Manhattan distance
manhattan = np.sum(np.abs(sample_a - sample_b))

print("Distance Metrics Example")
print("=" * 45)
print(f"Sample A: N={sample_a[0]}, Fines={sample_a[1]}%, WC={sample_a[2]}%")
print(f"Sample B: N={sample_b[0]}, Fines={sample_b[1]}%, WC={sample_b[2]}%")
print(f"\nEuclidean distance: {euclidean:.2f}")
print(f"Manhattan distance: {manhattan:.2f}")
print(f"\nNotice: Fines content difference (35) dominates because its range is larger.")
print(f"This is why we MUST scale features before using KNN!")

---

## Part 3: Prepare Data — Train/Test Split and Scaling

In [ ]:
# ─────────────────────────────────────────────────────
# PREPARE FEATURES AND TARGET
# ─────────────────────────────────────────────────────

feature_cols = ['N_value', 'fines_content', 'water_content', 'liquid_limit', 'plasticity_index']
X = df[feature_cols]
y = df['soil_type']

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nClass distribution in training set:\n{y_train.value_counts()}")

In [ ]:
# ─────────────────────────────────────────────────────
# FEATURE SCALING — StandardScaler
# ─────────────────────────────────────────────────────
# StandardScaler transforms each feature to have mean=0 and std=1.
# CRITICAL: Fit on training data ONLY, then transform both train and test.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform on training
X_test_scaled = scaler.transform(X_test)         # transform only on test

print("Before scaling (training set):")
print(f"  N_value range:       {X_train['N_value'].min():.0f} to {X_train['N_value'].max():.0f}")
print(f"  fines_content range: {X_train['fines_content'].min():.1f} to {X_train['fines_content'].max():.1f}")
print(f"\nAfter scaling (training set):")
print(f"  N_value range:       {X_train_scaled[:, 0].min():.2f} to {X_train_scaled[:, 0].max():.2f}")
print(f"  fines_content range: {X_train_scaled[:, 1].min():.2f} to {X_train_scaled[:, 1].max():.2f}")
print(f"\nAll features now have comparable scales — distances are fair.")

---

## Part 4: Train KNN Classifier

In [ ]:
# ─────────────────────────────────────────────────────
# TRAIN KNN MODEL
# ─────────────────────────────────────────────────────

knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = knn.predict(X_test_scaled)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"KNN Accuracy (K=5, Euclidean): {acc:.2%}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# ─────────────────────────────────────────────────────
# CONFUSION MATRIX
# ─────────────────────────────────────────────────────

labels = ['Clay', 'Gravel', 'Sand', 'Silt']
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted Soil Type')
ax.set_ylabel('Actual Soil Type')
ax.set_title('KNN Confusion Matrix (K=5)')

# Add text annotations
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)

plt.tight_layout()
plt.show()
print("Diagonal = correct predictions. Off-diagonal = misclassifications.")

---

## Part 5: Effect of K on Bias-Variance Tradeoff

- **Small K** (e.g., K=1): Model is very flexible, captures noise (high variance, low bias)
- **Large K** (e.g., K=50): Model is very smooth, may miss patterns (low variance, high bias)
- We need to find the **sweet spot** where test accuracy is maximized.

In [ ]:
# ─────────────────────────────────────────────────────
# EFFECT OF K — Accuracy vs. K
# ─────────────────────────────────────────────────────

k_range = range(1, 31)
train_scores = []
test_scores = []

for k in k_range:
    knn_k = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_k.fit(X_train_scaled, y_train)
    train_scores.append(knn_k.score(X_train_scaled, y_train))
    test_scores.append(knn_k.score(X_test_scaled, y_test))

best_k = k_range[np.argmax(test_scores)]
best_acc = max(test_scores)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, train_scores, 'b-o', markersize=4, label='Training Accuracy')
ax.plot(k_range, test_scores, 'r-s', markersize=4, label='Test Accuracy')
ax.axvline(x=best_k, color='green', linestyle='--', alpha=0.7, label=f'Best K={best_k}')
ax.set_xlabel('K (Number of Neighbors)')
ax.set_ylabel('Accuracy')
ax.set_title('KNN: Effect of K on Accuracy (Bias-Variance Tradeoff)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, 31, 2))
plt.tight_layout()
plt.show()

print(f"Best K = {best_k} with test accuracy = {best_acc:.2%}")
print(f"\nK=1: Train={train_scores[0]:.2%}, Test={test_scores[0]:.2%} (overfitting)")
print(f"K=30: Train={train_scores[-1]:.2%}, Test={test_scores[-1]:.2%} (underfitting)")

---

## Part 6: Comparing Distance Metrics

In [ ]:
# ─────────────────────────────────────────────────────
# COMPARE DISTANCE METRICS
# ─────────────────────────────────────────────────────

metrics = ['euclidean', 'manhattan', 'minkowski']
print(f"{'Metric':<15} {'Train Acc':>10} {'Test Acc':>10}")
print("─" * 38)

for metric in metrics:
    knn_m = KNeighborsClassifier(n_neighbors=best_k, metric=metric)
    knn_m.fit(X_train_scaled, y_train)
    train_acc = knn_m.score(X_train_scaled, y_train)
    test_acc = knn_m.score(X_test_scaled, y_test)
    print(f"{metric:<15} {train_acc:>10.2%} {test_acc:>10.2%}")

print(f"\nNote: Minkowski with p=2 (default) is identical to Euclidean.")
print(f"Manhattan can sometimes perform better when features have different distributions.")

---

## Part 7: Impact of Feature Scaling — With vs Without

In [ ]:
# ─────────────────────────────────────────────────────
# SCALING COMPARISON — Scaled vs Unscaled
# ─────────────────────────────────────────────────────

# Without scaling
knn_unscaled = KNeighborsClassifier(n_neighbors=best_k)
knn_unscaled.fit(X_train, y_train)  # raw features
acc_unscaled = knn_unscaled.score(X_test, y_test)

# With scaling
knn_scaled = KNeighborsClassifier(n_neighbors=best_k)
knn_scaled.fit(X_train_scaled, y_train)
acc_scaled = knn_scaled.score(X_test_scaled, y_test)

print("Feature Scaling Impact on KNN")
print("=" * 40)
print(f"Without scaling: {acc_unscaled:.2%}")
print(f"With scaling:    {acc_scaled:.2%}")
print(f"Improvement:     {acc_scaled - acc_unscaled:+.2%}")
print(f"\nFeature scaling is ESSENTIAL for KNN because it uses distances.")
print(f"Without scaling, features with larger ranges dominate the distance.")

---

## Part 8: Visualize Decision Boundaries (2D)

To visualize how KNN draws boundaries between soil types, we project to 2D using the two most informative features: **fines content** and **plasticity index**.

In [ ]:
# ─────────────────────────────────────────────────────
# DECISION BOUNDARY VISUALIZATION (2D)
# ─────────────────────────────────────────────────────

# Use 2 features for visualization
feat_x, feat_y = 'fines_content', 'plasticity_index'
X_2d = df[[feat_x, feat_y]].values
y_labels = df['soil_type'].values

# Scale 2D features
scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

# Encode labels to integers for coloring
soil_classes = ['Clay', 'Gravel', 'Sand', 'Silt']
label_to_int = {s: i for i, s in enumerate(soil_classes)}
y_int = np.array([label_to_int[s] for s in y_labels])

# Train KNN on 2D data
knn_2d = KNeighborsClassifier(n_neighbors=5)
knn_2d.fit(X_2d_scaled, y_int)

# Create mesh grid for decision boundary
h = 0.05  # step size
x_min, x_max = X_2d_scaled[:, 0].min() - 1, X_2d_scaled[:, 0].max() + 1
y_min, y_max = X_2d_scaled[:, 1].min() - 1, X_2d_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot
colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71']
cmap_bg = ListedColormap([c + '40' for c in colors])  # light background
cmap_pts = ListedColormap(colors)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, k_val in enumerate([1, 5, 15]):
    ax = axes[idx]
    knn_temp = KNeighborsClassifier(n_neighbors=k_val)
    knn_temp.fit(X_2d_scaled, y_int)
    Z = knn_temp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, levels=[-0.5, 0.5, 1.5, 2.5, 3.5], colors=colors)
    for i, soil in enumerate(soil_classes):
        mask = y_int == i
        ax.scatter(X_2d_scaled[mask, 0], X_2d_scaled[mask, 1],
                   c=colors[i], label=soil, edgecolors='black', s=50, alpha=0.8)
    ax.set_xlabel(f'{feat_x} (scaled)')
    ax.set_ylabel(f'{feat_y} (scaled)')
    ax.set_title(f'K = {k_val}')
    ax.legend(loc='upper left', fontsize=8)

fig.suptitle('KNN Decision Boundaries: Effect of K', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("K=1: Jagged boundaries (overfitting to individual points)")
print("K=5: Smoother boundaries (good balance)")
print("K=15: Very smooth boundaries (may lose local patterns)")

---

## Part 9: Predict New Soil Samples

In [ ]:
# ─────────────────────────────────────────────────────
# PREDICT NEW SAMPLES
# ─────────────────────────────────────────────────────

# Retrain best model on all features
knn_final = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn_final.fit(X_train_scaled, y_train)

# New soil samples from a site investigation
new_samples = pd.DataFrame({
    'N_value': [5, 30, 8, 45],
    'fines_content': [85.0, 5.0, 65.0, 3.0],
    'water_content': [40.0, 10.0, 25.0, 5.0],
    'liquid_limit': [55.0, 8.0, 30.0, 5.0],
    'plasticity_index': [28.0, 2.0, 8.0, 1.0]
})

new_scaled = scaler.transform(new_samples)
predictions = knn_final.predict(new_scaled)

print("Predictions for New Soil Samples")
print("=" * 65)
print(f"{'N':>4} {'Fines':>7} {'WC':>6} {'LL':>6} {'PI':>6}  {'Predicted':>10}")
print("─" * 65)
for i in range(len(new_samples)):
    row = new_samples.iloc[i]
    print(f"{row['N_value']:4.0f} {row['fines_content']:7.1f} {row['water_content']:6.1f} "
          f"{row['liquid_limit']:6.1f} {row['plasticity_index']:6.1f}  {predictions[i]:>10}")

print(f"\nKNN correctly identifies soil types from their geotechnical properties.")

---

## Part 10: Feature Importance Analysis

In [ ]:
# ─────────────────────────────────────────────────────
# FEATURE IMPORTANCE — Leave-One-Out approach
# ─────────────────────────────────────────────────────
# KNN has no built-in feature importance. We measure it by
# dropping one feature at a time and seeing how accuracy drops.

baseline_acc = knn_final.score(X_test_scaled, y_test)
importance = {}

for i, col in enumerate(feature_cols):
    # Zero out one feature in test data
    X_test_dropped = X_test_scaled.copy()
    X_test_dropped[:, i] = 0  # Set to mean (which is 0 after scaling)
    acc_dropped = knn_final.score(X_test_dropped, y_test)
    importance[col] = baseline_acc - acc_dropped

# Sort and plot
imp_sorted = dict(sorted(importance.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(list(imp_sorted.keys()), list(imp_sorted.values()), color='steelblue')
ax.set_xlabel('Accuracy Drop When Feature Removed')
ax.set_title('KNN Feature Importance (Permutation-style)')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("Feature Importance (accuracy drop when removed):")
for feat, imp in imp_sorted.items():
    print(f"  {feat:<20s}: {imp:+.4f}")
print(f"\nFines content and plasticity index are the strongest discriminators.")

---

## Practice Exercises

### Exercise 1: Try Different K Values
Train KNN with K=3 and K=20. Compare their accuracy on the test set. Which performs better and why?

### Exercise 2: Manhattan Distance
Retrain the model using Manhattan distance. Does it improve accuracy for soil classification?

### Exercise 3: Add Weighted Voting
Use `weights='distance'` in KNeighborsClassifier so closer neighbors have more influence. Compare with uniform weights.

In [ ]:
# EXERCISE 1: Try K=3 and K=20
# knn_3 = KNeighborsClassifier(n_neighbors=3)
# knn_3.fit(X_train_scaled, y_train)
# print(f"K=3 accuracy: {knn_3.score(X_test_scaled, y_test):.2%}")
# ... try K=20

In [ ]:
# EXERCISE 2: Manhattan distance
# knn_man = KNeighborsClassifier(n_neighbors=best_k, metric='manhattan')
# ...

In [ ]:
# EXERCISE 3: Distance-weighted voting
# knn_weighted = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
# ...

---

## Key Takeaways

1. **KNN classifies by majority vote** of the K nearest neighbors in feature space. It memorizes training data rather than learning a parametric model.

2. **Feature scaling is essential** for KNN because it uses distance calculations. Without scaling, features with larger ranges (like fines content 0-100%) dominate over features with smaller ranges (like N-value 0-60).

3. **K controls the bias-variance tradeoff**: Small K gives complex, noisy boundaries (overfitting); large K gives smooth boundaries (underfitting). Use cross-validation to find the optimal K.

4. **Decision boundaries** become smoother as K increases, which is visible in the 2D plots. For soil classification, moderate K values (3-7) often work best.

5. **KNN is simple but effective** for soil classification when features are well-separated. Its main weakness is slow prediction on large datasets (must compute all distances).

---

## How This Connects to ML

| What We Did Today | ML Equivalent |
|-------------------|---------------|
| Classify soil from properties | Multi-class classification |
| StandardScaler normalization | Feature preprocessing for distance-based models |
| Varying K from 1 to 30 | Hyperparameter tuning (bias-variance tradeoff) |
| Euclidean vs Manhattan | Distance metric selection |
| Confusion matrix | Classification evaluation |
| Drop-one-feature importance | Permutation feature importance |
| Decision boundary plots | Model interpretability |

---

*Day 040 of 100 — KNN: the simplest classifier that often works surprisingly well!*